# Entrenamiento de los modelos SSPNet (Google Colab, 2022)

Cuaderno con el que se entrenaron en Google Colab Pro los dos modelos SSPNet del TFG (SSPNet v2 y SSPNet Tiny v1). Las versiones fijadas en la instalación son las del Colab de 2022 (Python 3.7, PyTorch 1.9 con CUDA 11.1, mmcv-full 1.4). En 2026 se adaptaron las rutas y la instalación a la estructura del repositorio, pero el cuaderno no se ha vuelto a ejecutar: para evaluar o entrenar hoy, el entorno reproducible es el contenedor de `api/app/Dockerfile.dev` (ver el README).


## Instalaciones y setup


Aquí instalamos todo lo necesario para que el cuaderno Python funcione, esto es
la instalación de pytorch, mmdetection, mmcv y el repositorio donde está el trabajo realizado

In [ ]:
# install dependencies: because cuda 1.11cu113 not working for mmcv-full
!pip install torch==1.9.0+cu111 torchvision==0.10.0+cu111 -f https://download.pytorch.org/whl/torch_stable.html

# install mmcv-full thus we could use CUDA operators
!pip install mmcv-full==1.4.0 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.9.0/index.html

!pip install scikit-image

!git clone https://github.com/angtordom1/DPIApp

%cd DPIApp/api/app

# mmdet se usa desde api/app sin instalarlo como paquete: la ruta va en PYTHONPATH
# para los scripts de tools/ y en sys.path para este cuaderno
!pip install cython pycocotools six tqdm
%env PYTHONPATH=/content/DPIApp/api/app
import sys
sys.path.insert(0, '/content/DPIApp/api/app')

%cd huicv/deps/mini_maskrcnn_benchmark
!python setup.py build develop
from mini_maskrcnn_benchmark.layers import nms as _box_nms
%cd /content/DPIApp/api/app

!pip install -U albumentations
!pip install opencv-python==4.5.5.64
!pip install terminaltables
!python -m pip uninstall matplotlib
!pip install matplotlib==3.1.3

Para poder realizar los experimentos, necesitamos tener un zip con el dataset de TinyPerson en nuestro Drive. El zip contiene la carpeta `tiny/` (con `tiny_set/` y los pesos preentrenados de `params/`); se descomprime en `data/` y se renombra a `data/tinyPerson/`, que es donde lo buscan los configs. El dataset se descarga desde su fuente oficial, el repositorio [TinyBenchmark](https://github.com/ucas-vg/TinyBenchmark).

In [2]:
from google.colab import drive

drive.mount('/content/drive')

!mkdir -p data && unzip -q /content/drive/MyDrive/tiny -d data && mv data/tiny data/tinyPerson

Mounted at /content/drive


Por último, realizaremos las importaciones de todas las librerías necesarias

In [ ]:
# Imports
import matplotlib.pyplot as plt

import mmcv
from mmcv import Config
from mmcv.runner import load_checkpoint

from mmdet.apis import inference_detector, show_result_pyplot, set_random_seed, train_detector
from mmdet.models import build_detector
from mmdet.datasets import build_dataset

import os.path as osp

import torch, torchvision


## Entrenamiento

### SSPNet

In [ ]:
cfg = Config.fromfile('configs/models/faster_rcnn_r50_sspnet_1x.py')

cfg.model.pop('pretrained', None)
# Nota (2026): este checkpoint no llegó a cargarse. Los pesos del modelo entrenado muestran que
# partió de ResNet-50 de ImageNet, como SSPNet Tiny: muy probablemente porque el checkpoint usa los
# nombres de capa de maskrcnn_benchmark (module.backbone.body...), que MMDetection omite sin dar error.
cfg.model.init_cfg = dict(type='Pretrained',
            checkpoint='data/tinyPerson/params/COCOPretrain/FPN_msm_tinyperson_b4/model_only_model_final.pth')

# Dividimos por 4 debido a que solo estamos usando una GPU y MMDetection suele
# trabajar con 4
cfg.optimizer.lr = 0.01/4

# Establecemos el directorio donde se guardarán los checkpoints y logs
cfg.work_dir = './work_dirs/sspnet_1x'

# Establecemos el intervalo de evaluación, checkpoint y las épocas de 
# entrenamiento
cfg.evaluation.interval = 12
cfg.checkpoint_config.interval = 6
cfg.runner.max_epochs = 12

# Configuramos una semilla y decimos que vamos a usar una GPU
cfg.seed = 24
cfg.gpu_ids = range(1)
cfg.device='cuda'    

# Configuramos tensorboard para registrar el proceso de entrenamiento
cfg.log_config.hooks = [
    dict(type='TextLoggerHook'),
    dict(type='TensorboardLoggerHook')]

# Mostramos la configuración del modelo
print(f'Config:\n{cfg.pretty_text}')

In [ ]:
# Construimos el dataset y el detector
datasets = [build_dataset(cfg.data.train)]
model = build_detector(cfg.model)
model.CLASSES = datasets[0].CLASSES

# Creamos el directorio de trabajo
mmcv.mkdir_or_exist(osp.abspath(cfg.work_dir))
train_detector(model, datasets, cfg, distributed=False, validate=True)

A continuación, se verá tanto la visualización del entrenamiento recogida con tensorboard como el test

In [ ]:
# Cargamos tensorboard
%load_ext tensorboard

# Vemos los resultados
%tensorboard --logdir ./work_dirs/sspnet_1x

In [ ]:
# Test sobre bbox
!python tools/test.py \
    configs/models/faster_rcnn_r50_sspnet_1x.py \
    work_dirs/sspnet_1x/epoch_12.pth \
    --eval bbox

Ahora, podemos ver los resultados del modelo entrenado

In [ ]:
from PIL import Image
import glob
import random
from mmdet.apis import inference_detector,show_result_pyplot

random.seed(0)
model.cfg = cfg
image_list = []

for filename in glob.glob('./data/tinyPerson/tiny_set/test/no_label_images/*.jpg'):
    image_list.append(filename)

for image in random.choices(image_list, k=10):
  img = mmcv.imread(image)
  result = inference_detector(model, img)
  show_result_pyplot(model, img, result, score_thr=0.7)

Por último, publicamos el modelo para utilizarlo en el servidor.

In [ ]:
!python tools/model_converters/publish_model.py work_dirs/sspnet_1x/epoch_12.pth faster_rcnn_r50_sspnet_1x.pth

### SSPNet Tiny

In [ ]:
cfg = Config.fromfile('configs/models/faster_rcnn_r50_sspnet_1x_Tiny.py')

# Dividimos por 4 debido a que solo estamos usando una GPU y MMDetection suele
# trabajar con 4
cfg.optimizer.lr = 0.01/4

# Establecemos el directorio donde se guardarán los checkpoints y logs
cfg.work_dir = './work_dirs/sspnet_1x_Tiny'

# Establecemos el intervalo de evaluación, checkpoint y las épocas de 
# entrenamiento
cfg.evaluation.interval = 12
cfg.checkpoint_config.interval = 6
cfg.runner.max_epochs = 12

# Configuramos una semilla y decimos que vamos a usar una GPU
cfg.seed = 24
cfg.gpu_ids = range(1)
cfg.device='cuda'    

# Configuramos tensorboard para registrar el proceso de entrenamiento
cfg.log_config.hooks = [
    dict(type='TextLoggerHook'),
    dict(type='TensorboardLoggerHook')]

# Mostramos la configuración del modelo
print(f'Config:\n{cfg.pretty_text}')

In [ ]:
# Construimos el dataset y el detector
datasets = [build_dataset(cfg.data.train)]
model = build_detector(cfg.model)
model.CLASSES = datasets[0].CLASSES

# Creamos el directorio de trabajo
mmcv.mkdir_or_exist(osp.abspath(cfg.work_dir))
train_detector(model, datasets, cfg, distributed=False, validate=True)

A continuación, se verá tanto la visualización del entrenamiento recogida con tensorboard como el test

In [ ]:
# Cargamos tensorboard
%load_ext tensorboard

# Vemos los resultados
%tensorboard --logdir ./work_dirs/sspnet_1x_Tiny

In [ ]:
# Test sobre bbox
!python tools/test.py \
    configs/models/faster_rcnn_r50_sspnet_1x_Tiny.py \
    work_dirs/sspnet_1x_Tiny/epoch_12.pth \
    --eval bbox

Ahora, podemos ver los resultados del modelo entrenado

In [ ]:
from PIL import Image
import glob
import random
from mmdet.apis import inference_detector,show_result_pyplot

random.seed(0)
model.cfg = cfg
image_list = []

for filename in glob.glob('./data/tinyPerson/tiny_set/test/no_label_images/*.jpg'):
    image_list.append(filename)

for image in random.choices(image_list, k=10):
  img = mmcv.imread(image)
  result = inference_detector(model, img)
  show_result_pyplot(model, img, result, score_thr=0.7)

Por último, publicamos el modelo para utilizarlo en el servidor.

In [ ]:
!python tools/model_converters/publish_model.py work_dirs/sspnet_1x_Tiny/epoch_12.pth faster_rcnn_r50_sspnet_1x_Tiny.pth